In [ ]:
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import squidpy as sq
from scipy.ndimage import gaussian_filter
from skimage.color import label2rgb
from skimage.draw import ellipse
from skimage.measure import label, regionprops, regionprops_table
from skimage.transform import rotate
import warnings

from scipy.stats import kruskal
import scikit_posthocs as sp
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from skimage.segmentation import expand_labels, watershed

import os
import cv2
from skimage.feature import graycomatrix, graycoprops


In [ ]:
root_path = 'patch_path'
all_results = {}

for tma_dir in os.listdir(root_path):
    tma_path = os.path.join(root_path, tma_dir)
    print(tma_path)
    if not os.path.isdir(tma_path):
        continue

    patches = [f for f in os.listdir(tma_path) if f.endswith('.npy')]
    # Store results for this TMA only
    tma_props = []

    for patch_file in patches:
        print(patch_file)
        patch_path = os.path.join(tma_path, patch_file)
        patch = np.load(patch_path, allow_pickle=True)
        # Normalize patch
        normalized_img = cv2.normalize(
            patch,
            None,
            0,
            255,
            cv2.NORM_MINMAX
        ).astype(np.uint8)
        # Convert to grayscale
        gray_image = cv2.cvtColor(
            normalized_img,
            cv2.COLOR_RGB2GRAY
        )

        # GLCM
        glcm = graycomatrix(
            gray_image,
            distances=[1],
            angles=[0],
            levels=256,
            symmetric=True,
            normed=True
        )
        # GLCM properties
        contrast = graycoprops(glcm, 'contrast')[0, 0]
        dissimilarity = graycoprops(glcm, 'dissimilarity')[0, 0]
        homogeneity = graycoprops(glcm, 'homogeneity')[0, 0]
        energy = graycoprops(glcm, 'energy')[0, 0]
        correlation = graycoprops(glcm, 'correlation')[0, 0]
        ASM = graycoprops(glcm, 'ASM')[0, 0]

        # Basic intensity properties
        mean = np.mean(gray_image)
        variance = np.var(gray_image)
        std = np.std(gray_image)

        # Entropy
        histogram = np.bincount(
            gray_image.ravel(),
            minlength=256
        )
        probability = histogram / histogram.sum()
        probability = probability[probability > 0]

        entropy = -np.sum(
            probability * np.log2(probability)
        )

        # Create dictionary
        props = {
            'patch_name': patch_file,
            'tma': tma_dir,
            'contrast': contrast,
            'dissimilarity': dissimilarity,
            'homogeneity': homogeneity,
            'energy': energy,
            'correlation': correlation,
            'ASM': ASM,
            'mean': mean,
            'variance': variance,
            'std': std,
            'entropy': entropy
        }

        tma_props.append(props)

    # Create dataframe for this TMA
    tma_df = pd.DataFrame(tma_props)
    all_results[tma_dir] = tma_df
    print(f"{tma_dir}: {len(tma_df)} patches analyzed")


# Combine all TMAs
master_df = pd.concat(
    all_results.values(),
    ignore_index=True
)

print(f"\nTotal patches: {len(master_df)}")